Hypothesis
-----
Does per-neuron gradient variance differ between an overfitting
model and a generalizing (regularized) model?

Setup
-----
- Data: sklearn make_moons, small N, with noise -> easy to overfit with a
  capacity-heavy MLP, but also learnable with proper regularization.
- Model A ("overfit"): no regularization, no dropout, trained blindly for
  all 500 epochs on the full training set, sees no validation signal during
  training (mirrors "crank epochs with nothing to stop you").
- Model B ("regularized"): same architecture, but with dropout + L2 weight
  decay, and we track val loss (model still trains all 1000 epochs so the
  epoch axes line up for comparison -- we are NOT early-stopping it, just
  regularizing it, so the two gradient traces are directly comparable epoch
  for epoch).
- Both models use IDENTICAL weight initialization (same seed, same
  architecture) so neuron index N in model A and neuron index N in model B
  start out as literally the same neuron.
- We log the L2 norm and angle of the gradient flowing into each probed neuron's
  incoming weight vector at every epoch.

Findings
-----
- We consistently observe higher variance in gradients in neurons of the overfitted model after
  it enters overfitting territory (measured as when training accuracy significantly
  overtakes validation accuracy ~ epoch 200)
- 95.5% (+/- 4.2%) neurons in hidden layer 1 displayed a lower mean cosine similarity
  of gradient vectors as calculated between epoch t and epoch t-1 for all epochs > 200.
- 86.6% (+/- 7.4%)  neurons in hidden layer 1 displayed a higher normalized variance in
  gradient magnitudes in the overfitted model as compared to the generalizing baseline.
- The baseline also included a few dead neurons with gradient mean ~ 0. These were eliminated
  before further calculation.

Next Steps
-----
- Firstly, replicate results with multiple datasets to validate this claim.
- Isolate impact of overfitting on gradient variance by splitting data into
  multiple random samples and checking whether samples agree or disagree on
  the gradient. This eliminates noise in variance due to model architecture
  and data idiosyncracies.

In [37]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import make_moons
import json
import matplotlib.pyplot as plt

In [191]:
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

N_EPOCHS = 1000
HIDDEN = 128            # very wide relative to dataset size -> easy to overfit
N_SAMPLES = 100          # small training set -> easy to overfit
NOISE = 0.5            # noisy boundary -> generalizable signal is genuinely harder
LABEL_FLIP_FRAC = 0.1  # fraction of TRAIN labels flipped -> idiosyncratic noise to memorize
NEURON_LAYER = "fc1"    # which layer we probe (first hidden layer, width=HIDDEN)
N_PROBE_NEURONS = 128

In [193]:
# ----------------------------------------------------------------------------
# 1. Data
# ----------------------------------------------------------------------------
# Train and val are drawn as separate make_moons samples (different random
# draws) rather than splitting one pool, so val is a genuine generalization
# test rather than an easy held-out slice of the same tight clusters.
X_train, y_train = make_moons(n_samples=N_SAMPLES, noise=NOISE, random_state=SEED)
X_val, y_val = make_moons(n_samples=300, noise=NOISE, random_state=SEED + 1)

# Inject label noise into the TRAINING set only. This gives the overfit
# model concrete idiosyncratic points it can only fit by memorizing rather
# than by learning the true moon-shaped boundary.
rng_flip = np.random.default_rng(SEED)
n_flip = int(LABEL_FLIP_FRAC * len(y_train))
flip_idx = rng_flip.choice(len(y_train), size=n_flip, replace=False)
y_train = y_train.copy()
y_train[flip_idx] = 1 - y_train[flip_idx]
print(f"Flipped {n_flip}/{len(y_train)} training labels to inject memorizable noise.")

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)

print(f"Train size: {len(X_train)}, Val size: {len(X_val)}")

Flipped 10/100 training labels to inject memorizable noise.
Train size: 100, Val size: 300


In [195]:
# ----------------------------------------------------------------------------
# 2. Model definition (identical architecture for both models)
# ----------------------------------------------------------------------------
class MLP(nn.Module):
    def __init__(self, hidden=HIDDEN, dropout_p=0.0):
        super().__init__()
        self.fc1 = nn.Linear(2, hidden)
        self.act1 = nn.ReLU()
        self.drop1 = nn.Dropout(dropout_p) if dropout_p > 0 else nn.Identity()
        self.fc2 = nn.Linear(hidden, hidden)
        self.act2 = nn.ReLU()
        self.drop2 = nn.Dropout(dropout_p) if dropout_p > 0 else nn.Identity()
        self.fc3 = nn.Linear(hidden, 1)

    def forward(self, x):
        x = self.act1(self.fc1(x))
        x = self.drop1(x)
        x = self.act2(self.fc2(x))
        x = self.drop2(x)
        x = self.fc3(x)
        return x


def make_model(dropout_p):
    torch.manual_seed(SEED)  # identical init for both models
    return MLP(hidden=HIDDEN, dropout_p=dropout_p)


model_overfit = make_model(dropout_p=0.0)
model_reg = make_model(dropout_p=0.0)

# sanity check: identical initial weights on fc1
assert torch.allclose(model_overfit.fc1.weight, model_reg.fc1.weight)
print("Confirmed: both models start with identical fc1 weights.")

# Pick fixed neuron indices in fc1 (same neurons probed in both models)
rng = np.random.default_rng(SEED)
probe_indices = sorted(rng.choice(HIDDEN, size=N_PROBE_NEURONS, replace=False).tolist())
print(f"Probing neuron indices in {NEURON_LAYER}: {probe_indices}")

Confirmed: both models start with identical fc1 weights.
Probing neuron indices in fc1: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127]


In [197]:
# ----------------------------------------------------------------------------
# 3. Training loop with per-neuron gradient logging
# ----------------------------------------------------------------------------
def train_and_log(model, weight_decay, label):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=weight_decay)

    grad_log = {idx: [] for idx in probe_indices}
    grad_vec_log = {idx: [] for idx in probe_indices}
    train_losses, val_losses = [], []
    train_accs, val_accs = [], []

    target_layer_weight = dict(model.named_parameters())[f"{NEURON_LAYER}.weight"]

    for epoch in range(N_EPOCHS):
        model.train()
        optimizer.zero_grad()
        logits = model(X_train_t)
        loss = criterion(logits, y_train_t)
        loss.backward()

        # log gradient norm for each probed neuron's incoming weight vector
        # target_layer_weight.grad has shape [HIDDEN, 2] for fc1
        grad = target_layer_weight.grad.detach()
        for idx in probe_indices:
            neuron_grad_norm = grad[idx].norm().item()
            grad_log[idx].append(neuron_grad_norm)
            grad_vec_log[idx].append(grad[idx].tolist())

        optimizer.step()

        with torch.no_grad():
            train_pred = (torch.sigmoid(logits) > 0.5).float()
            train_acc = (train_pred == y_train_t).float().mean().item()
            train_losses.append(loss.item())
            train_accs.append(train_acc)

            model.eval()
            val_logits = model(X_val_t)
            val_loss = criterion(val_logits, y_val_t).item()
            val_pred = (torch.sigmoid(val_logits) > 0.5).float()
            val_acc = (val_pred == y_val_t).float().mean().item()
            val_losses.append(val_loss)
            val_accs.append(val_acc)

        if epoch % 100 == 0 or epoch == N_EPOCHS - 1:
            print(
                f"[{label}] epoch {epoch:4d} | train_loss {loss.item():.4f} "
                f"val_loss {val_loss:.4f} | train_acc {train_acc:.3f} val_acc {val_acc:.3f}"
            )

    return {
        "grad_log": grad_log,
        "grad_vec_log": grad_vec_log,
        "train_losses": train_losses,
        "val_losses": val_losses,
        "train_accs": train_accs,
        "val_accs": val_accs,
    }


print("\n--- Training OVERFIT model (no regularization) ---")
results_overfit = train_and_log(model_overfit, weight_decay=0.0, label="overfit")

print("\n--- Training REGULARIZED model (dropout + weight decay) ---")
results_reg = train_and_log(model_reg, weight_decay=5e-3, label="regularized")



--- Training OVERFIT model (no regularization) ---
[overfit] epoch    0 | train_loss 0.7081 val_loss 0.6889 | train_acc 0.450 val_acc 0.520
[overfit] epoch  100 | train_loss 0.4398 val_loss 0.4390 | train_acc 0.810 val_acc 0.787
[overfit] epoch  200 | train_loss 0.3690 val_loss 0.4612 | train_acc 0.810 val_acc 0.797
[overfit] epoch  300 | train_loss 0.3058 val_loss 0.5176 | train_acc 0.840 val_acc 0.757
[overfit] epoch  400 | train_loss 0.2492 val_loss 0.5985 | train_acc 0.900 val_acc 0.750
[overfit] epoch  500 | train_loss 0.1917 val_loss 0.6978 | train_acc 0.950 val_acc 0.743
[overfit] epoch  600 | train_loss 0.1446 val_loss 0.8290 | train_acc 0.970 val_acc 0.743
[overfit] epoch  700 | train_loss 0.1090 val_loss 0.9590 | train_acc 0.970 val_acc 0.737
[overfit] epoch  800 | train_loss 0.0832 val_loss 1.1013 | train_acc 0.980 val_acc 0.740
[overfit] epoch  900 | train_loss 0.0642 val_loss 1.2410 | train_acc 0.990 val_acc 0.743
[overfit] epoch  999 | train_loss 0.0488 val_loss 1.3642 |

In [198]:
# ----------------------------------------------------------------------------
# 4. Save raw results to disk (so plotting can be re-run without retraining)
# ----------------------------------------------------------------------------
filename = f"results_{SEED}.json"
with open(filename, "w") as f:
    json.dump(
        {
            "probe_indices": probe_indices,
            "overfit": results_overfit,
            "regularized": results_reg,
        },
        f,
    )

print("\nSaved results to ")
print(filename)


Saved results to 
results_42.json


In [51]:
with open(f"results_{SEED}.json", "r") as f:
    data = json.load(f)

probe_indices = data["probe_indices"]
overfit = data["overfit"]
reg = data["regularized"]

N_EPOCHS = len(overfit["train_losses"])
epochs = np.arange(N_EPOCHS)

# ----------------------------------------------------------------------------
# Figure 1: train/val loss curves for both models (context for what follows)
# ----------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(epochs, overfit["train_losses"], label="train loss", color="#d62728")
axes[0].plot(epochs, overfit["val_losses"], label="val loss", color="#1f77b4")
axes[0].set_title("Overfit model (no regularization)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("BCE Loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(epochs, reg["train_losses"], label="train loss", color="#d62728")
axes[1].plot(epochs, reg["val_losses"], label="val loss", color="#1f77b4")
axes[1].set_title("Regularized model (dropout + L2)")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("BCE Loss")
axes[1].legend()
axes[1].grid(alpha=0.3)

fig.suptitle("Train vs. Validation Loss: Overfit vs. Regularized Model", fontsize=13)
fig.tight_layout()
fig.savefig(f"loss_curves_{SEED}.png", dpi=150)
plt.close(fig)

# ----------------------------------------------------------------------------
# Figure 2: 5 superimposed per-neuron gradient norm comparison charts
# ----------------------------------------------------------------------------
fig, axes = plt.subplots(len(probe_indices), 1, figsize=(11, 3.2 * len(probe_indices)), sharex=True)

for ax, idx in zip(axes, probe_indices):
    g_overfit = overfit["grad_log"][str(idx)]
    g_reg = reg["grad_log"][str(idx)]

    ax.plot(epochs, g_overfit, label="Overfit model", color="#d62728", alpha=0.85, linewidth=1.1)
    ax.plot(epochs, g_reg, label="Regularized model", color="#1f77b4", alpha=0.85, linewidth=1.1)
    ax.set_title(f"fc1 neuron #{idx} — incoming weight gradient norm")
    ax.set_ylabel("||grad||_2")
    ax.grid(alpha=0.3)
    ax.legend(loc="upper right", fontsize=8)

axes[-1].set_xlabel("Epoch")
fig.suptitle("Per-Neuron Gradient Norm Over Training: Overfit vs. Regularized\n(same 5 fixed neurons, same initialization, both models)", fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(f"neuron_gradients_comparison_{SEED}.png", dpi=150)
plt.close(fig)

# ----------------------------------------------------------------------------
# Figure 3: windowed variance of those same gradients (the actual hypothesis test)
# ----------------------------------------------------------------------------
WINDOW = 20

def windowed_variance(series, window):
    series = np.array(series)
    out = np.full_like(series, np.nan, dtype=float)
    for i in range(window, len(series)):
        out[i] = series[i - window : i].var()
    return out

fig, axes = plt.subplots(len(probe_indices), 1, figsize=(11, 3.2 * len(probe_indices)), sharex=True)

for ax, idx in zip(axes, probe_indices):
    g_overfit = overfit["grad_log"][str(idx)]
    g_reg = reg["grad_log"][str(idx)]

    wv_overfit = windowed_variance(g_overfit, WINDOW)
    wv_reg = windowed_variance(g_reg, WINDOW)

    ax.plot(epochs, wv_overfit, label="Overfit model", color="#d62728", alpha=0.85, linewidth=1.1)
    ax.plot(epochs, wv_reg, label="Regularized model", color="#1f77b4", alpha=0.85, linewidth=1.1)
    ax.set_title(f"fc1 neuron #{idx} — windowed (w={WINDOW}) gradient variance")
    ax.set_ylabel("Var(grad norm)")
    ax.grid(alpha=0.3)
    ax.legend(loc="upper right", fontsize=8)

axes[-1].set_xlabel("Epoch")
fig.suptitle(f"Windowed Gradient Variance (window={WINDOW} epochs): Overfit vs. Regularized", fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(f"neuron_gradient_variance_comparison_{SEED}.png", dpi=150)
plt.close(fig)

print("Saved: loss_curves.png, neuron_gradients_comparison.png, neuron_gradient_variance_comparison.png")

Saved: loss_curves.png, neuron_gradients_comparison.png, neuron_gradient_variance_comparison.png


In [201]:
DEAD_THRESHOLD = 0.0001

overfit_higher_raw = 0
overfit_higher_cv = 0
dead_neurons = []
n_valid = 0

for idx in probe_indices:
    g_overfit = np.array(overfit["grad_log"][str(idx)])
    g_reg = np.array(reg["grad_log"][str(idx)])

    post_200_overfit = g_overfit[200:]
    post_200_reg = g_reg[200:]

    mean_overfit = post_200_overfit.mean()
    mean_reg = post_200_reg.mean()

    # exclude dead neurons: either model's mean grad below threshold
    if mean_overfit < DEAD_THRESHOLD or mean_reg < DEAD_THRESHOLD:
        dead_neurons.append(idx)
        continue

    n_valid += 1

    if post_200_overfit.var() > post_200_reg.var():
        overfit_higher_raw += 1

    cv_overfit_val = post_200_overfit.std() / mean_overfit
    cv_reg_val = post_200_reg.std() / mean_reg
    if cv_overfit_val > cv_reg_val:
        overfit_higher_cv += 1

print(f"Random seed used: {SEED}")
print(f"Raw variance: overfit higher in {overfit_higher_raw}/{n_valid} neurons")
print(f"Normalized (CV): overfit higher in {overfit_higher_cv}/{n_valid} neurons")
print(f"Dead neurons excluded: {len(dead_neurons)}/{len(probe_indices)} -> {dead_neurons}")
print(f"Valid neurons compared: {n_valid}/{len(probe_indices)}")

Random seed used: 42
Raw variance: overfit higher in 94/102 neurons
Normalized (CV): overfit higher in 78/102 neurons
Dead neurons excluded: 26/128 -> [1, 7, 12, 18, 19, 28, 30, 31, 33, 35, 38, 42, 51, 55, 56, 60, 73, 80, 83, 87, 92, 97, 98, 110, 113, 120]
Valid neurons compared: 102/128


In [203]:
flipped = []
dead_neurons = []
n_valid = 0

for idx in probe_indices:
    g_overfit = np.array(overfit["grad_log"][str(idx)])[200:]
    g_reg = np.array(reg["grad_log"][str(idx)])[200:]

    mean_overfit = g_overfit.mean()
    mean_reg = g_reg.mean()

    if mean_overfit < DEAD_THRESHOLD or mean_reg < DEAD_THRESHOLD:
        dead_neurons.append(idx)
        continue

    n_valid += 1

    raw_says_overfit_higher = g_overfit.var() > g_reg.var()
    cv_overfit = g_overfit.std() / mean_overfit
    cv_reg = g_reg.std() / mean_reg
    cv_says_overfit_higher = cv_overfit > cv_reg

    if raw_says_overfit_higher and not cv_says_overfit_higher:
        flipped.append((idx, mean_overfit, mean_reg))

print(f"Dead neurons excluded: {len(dead_neurons)}/{len(probe_indices)} -> {dead_neurons}")
print(f"Valid neurons compared: {n_valid}/{len(probe_indices)}")
print(f"{len(flipped)} neurons flipped after normalizing")
for idx, mean_o, mean_r in flipped:
    print(f"  neuron {idx}: overfit mean grad = {mean_o:.5f}, reg mean grad = {mean_r:.5f}")

Dead neurons excluded: 26/128 -> [1, 7, 12, 18, 19, 28, 30, 31, 33, 35, 38, 42, 51, 55, 56, 60, 73, 80, 83, 87, 92, 97, 98, 110, 113, 120]
Valid neurons compared: 102/128
18 neurons flipped after normalizing
  neuron 0: overfit mean grad = 0.01443, reg mean grad = 0.00218
  neuron 6: overfit mean grad = 0.00204, reg mean grad = 0.00119
  neuron 23: overfit mean grad = 0.00343, reg mean grad = 0.00014
  neuron 25: overfit mean grad = 0.00818, reg mean grad = 0.00174
  neuron 36: overfit mean grad = 0.00404, reg mean grad = 0.00202
  neuron 43: overfit mean grad = 0.00055, reg mean grad = 0.00034
  neuron 52: overfit mean grad = 0.00495, reg mean grad = 0.00117
  neuron 61: overfit mean grad = 0.00199, reg mean grad = 0.00135
  neuron 63: overfit mean grad = 0.00706, reg mean grad = 0.00027
  neuron 67: overfit mean grad = 0.00706, reg mean grad = 0.00349
  neuron 70: overfit mean grad = 0.00941, reg mean grad = 0.00010
  neuron 75: overfit mean grad = 0.01604, reg mean grad = 0.00477
  

In [205]:
with open(f"results_{SEED}.json", "r") as f:
    data = json.load(f)

probe_indices = data["probe_indices"]
overfit = data["overfit"]
reg = data["regularized"]

num_epochs_logged = len(overfit["train_losses"])
epochs = np.arange(num_epochs_logged)

def cosine_similarity_series(grad_vecs):
    """grad_vecs: list of [g_x, g_y] per epoch. Returns cosine similarity
    between each epoch's gradient vector and the PREVIOUS epoch's, length
    N_EPOCHS-1 (first epoch has no predecessor)."""
    vecs = np.array(grad_vecs)  # shape [N_EPOCHS, 2]
    sims = np.full(len(vecs), np.nan)
    for t in range(1, len(vecs)):
        v_prev, v_curr = vecs[t - 1], vecs[t]
        norm_prev, norm_curr = np.linalg.norm(v_prev), np.linalg.norm(v_curr)
        if norm_prev > 1e-12 and norm_curr > 1e-12:
            sims[t] = np.dot(v_prev, v_curr) / (norm_prev * norm_curr)
    return sims

def angle_series(grad_vecs):
    """Returns the angle (radians) of the gradient vector at each epoch."""
    vecs = np.array(grad_vecs)
    return np.arctan2(vecs[:, 1], vecs[:, 0])

# ----------------------------------------------------------------------------
# Filter out dead neurons (same rule as before: either model's mean grad
# norm in the post-200 window below threshold)
# ----------------------------------------------------------------------------
valid_indices = []
for idx in probe_indices:
    mean_overfit = np.array(overfit["grad_log"][str(idx)])[200:].mean()
    mean_reg = np.array(reg["grad_log"][str(idx)])[200:].mean()
    if mean_overfit >= DEAD_THRESHOLD and mean_reg >= DEAD_THRESHOLD:
        valid_indices.append(idx)

print(f"Valid (non-dead) neurons for direction analysis: {len(valid_indices)}/{len(probe_indices)}")

# ----------------------------------------------------------------------------
# Plot 1: cosine similarity to previous epoch, for a handful of neurons
# ----------------------------------------------------------------------------
plot_indices = valid_indices

fig, axes = plt.subplots(len(plot_indices), 1, figsize=(11, 3.2 * len(plot_indices)), sharex=True)
for ax, idx in zip(axes, plot_indices):
    sim_overfit = cosine_similarity_series(overfit["grad_vec_log"][str(idx)])
    sim_reg = cosine_similarity_series(reg["grad_vec_log"][str(idx)])

    ax.plot(epochs, sim_overfit, label="Overfit model", color="#d62728", alpha=0.8, linewidth=0.9)
    ax.plot(epochs, sim_reg, label="Generalizing model", color="#1f77b4", alpha=0.8, linewidth=0.9)
    ax.axhline(0, color="gray", linewidth=0.5, linestyle="--")
    ax.set_title(f"fc1 neuron #{idx} — gradient direction consistency (cos sim, epoch t vs t-1)")
    ax.set_ylabel("cosine similarity")
    ax.set_ylim(-1.1, 1.1)
    ax.grid(alpha=0.3)
    ax.legend(loc="lower right", fontsize=8)

axes[-1].set_xlabel("Epoch")
fig.suptitle(
    "Gradient Direction Consistency: does the gradient keep pointing the same way?\n"
    "(1 = same direction as last epoch, 0 = unrelated, -1 = flipped)",
    fontsize=12,
)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(f"neuron_gradient_direction_consistency_{SEED}.png", dpi=150)
plt.close(fig)

# ----------------------------------------------------------------------------
# Summary stat: average cosine similarity post-epoch-200, across ALL valid
# neurons, for both models -- the direct test of "overfit model disagrees
# with itself over time more than the generalizing model does"
# ----------------------------------------------------------------------------
overfit_lower_consistency = 0
print("Mean Overfit Similarity | Mean Regular Similarity\n")
for idx in valid_indices:
    sim_overfit = cosine_similarity_series(overfit["grad_vec_log"][str(idx)])[200:]
    sim_reg = cosine_similarity_series(reg["grad_vec_log"][str(idx)])[200:]
    mean_sim_overfit = np.nanmean(sim_overfit)
    mean_sim_reg = np.nanmean(sim_reg)
    if mean_sim_overfit < mean_sim_reg:
        overfit_lower_consistency += 1
print(f"\nFor seed = {SEED}")
print(f"Overfit model shows LOWER direction consistency (more disagreement) "
      f"in {overfit_lower_consistency}/{len(valid_indices)} neurons post-epoch-200")

print("Saved: neuron_gradient_direction_consistency.png")

Valid (non-dead) neurons for direction analysis: 112/128
Mean Overfit Similarity | Mean Regular Similarity


For seed = 42
Overfit model shows LOWER direction consistency (more disagreement) in 98/112 neurons post-epoch-200
Saved: neuron_gradient_direction_consistency.png
